# ⭐ Day 43: Geospatial EDA on NYC Taxi Data  
## 🗺️ Maps, Locations & Spatial Insights for AI & ML
### Day 43 of 369-day Python & AI Learning Path 📍

## Welcome to Day 43! 🚕

Today we embark on an exciting journey into the world of **geospatial data analysis** using one of the most famous datasets in the data science community: the **NYC Taxi Trip dataset**. Geospatial analysis is a critical skill in modern AI and machine learning, with applications spanning ride-sharing optimization, delivery logistics, urban planning, and location-based predictive modeling.

Understanding spatial patterns allows us to uncover hidden insights about human behavior, traffic flows, and urban dynamics. Whether you're building a ride-sharing app, optimizing delivery routes, or predicting demand patterns, geospatial EDA forms the foundation of location-intelligent systems.

In this notebook, we'll learn how to:
- 📊 Analyze spatial distributions and patterns
- 🗺️ Create stunning map visualizations
- 📍 Identify hotspots and popular zones
- ⏰ Combine temporal and spatial insights
- 🤖 Extract features for geospatial machine learning models

Let's dive into the data and start exploring the concrete jungle where dreams are made of! 🌆

## 📋 Table of Contents

1. [Introduction to Geospatial Data and NYC Taxi Problem](#1-introduction-to-geospatial-data-and-nyc-taxi-problem)
2. [Loading the Dataset and Initial Inspection](#2-loading-the-dataset-and-initial-inspection)
3. [Understanding Key Spatial Features](#3-understanding-key-spatial-features)
4. [Basic Spatial Statistics](#4-basic-spatial-statistics)
5. [Visualizing Pickup and Dropoff Locations](#5-visualizing-pickup-and-dropoff-locations)
6. [Interactive Maps with Plotly](#6-interactive-maps-with-plotly)
7. [Analyzing Popular Pickup/Dropoff Zones](#7-analyzing-popular-pickupdropoff-zones)
8. [Trip Distance and Fare Analysis by Location](#8-trip-distance-and-fare-analysis-by-location)
9. [Temporal + Spatial Patterns](#9-temporal--spatial-patterns)
10. [Key Insights for Geospatial Modeling](#10-key-insights-for-geospatial-modeling)
11. [🛠️ Hands-On Exercises](#-hands-on-exercises)
12. [Solutions & Key Insights](#solutions--key-insights)

## 1. Introduction to Geospatial Data and NYC Taxi Problem 🗺️

**Geospatial data** refers to information that has a geographic component, typically represented as coordinates (latitude and longitude) or shapes (points, lines, polygons). In the context of NYC taxi data, each trip is defined by:

- **Pickup location** (latitude, longitude)
- **Dropoff location** (latitude, longitude)
- **Timestamps** (when the trip started and ended)
- **Trip metrics** (distance, fare, passenger count)

### Why Geospatial Analysis Matters for AI/ML:
1. **Demand Prediction**: Predict where and when ride requests will spike
2. **Dynamic Pricing**: Adjust fares based on location-specific demand
3. **Route Optimization**: Find optimal paths considering traffic patterns
4. **Fleet Management**: Position vehicles in high-demand zones
5. **Anomaly Detection**: Identify unusual trips or locations

### NYC Taxi Dataset Overview:
The NYC Taxi & Limousine Commission (TLC) releases detailed trip records including:
- Over 1 billion trips annually
- Precise GPS coordinates
- Fare and payment information
- Temporal data at minute-level granularity

Let's start by loading our data subset! 📊

In [ ]:
# Import essential libraries for geospatial analysis
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# Set visualization style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

# For interactive maps
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# For geospatial calculations
from math import radians, cos, sin, asin, sqrt

print("✅ All libraries imported successfully!")
print("📦 Ready for geospatial analysis!")

## 2. Loading the Dataset and Initial Inspection 📂

We'll work with a realistic subset of the NYC Taxi dataset containing 10,000 trips with full spatial and temporal information.

In [ ]:
# Create a realistic NYC Taxi dataset subset
# In practice, you would load: df = pd.read_csv('nyc_taxi_data.csv')
# For this educational notebook, we'll generate realistic data

np.random.seed(43)  # For reproducibility
n_samples = 10000

# NYC bounding box (approximate)
# Manhattan: lat 40.70-40.88, lon -74.02 to -73.93
# Brooklyn: lat 40.57-40.74, lon -74.04 to -73.83
# Queens: lat 40.49-40.80, lon -73.96 to -73.70
# Bronx: lat 40.79-40.91, lon -73.93 to -73.76

def generate_nyc_coordinates(n, center_lat=40.75, center_lon=-73.95, spread=0.15):
    """Generate realistic NYC coordinates with clustering around Manhattan"""
    # Use normal distribution centered on Manhattan with some spread
    lats = np.random.normal(center_lat, spread/3, n)
    lons = np.random.normal(center_lon, spread/3, n)
    
    # Clip to realistic NYC bounds
    lats = np.clip(lats, 40.48, 40.92)
    lons = np.clip(lons, -74.25, -73.70)
    return lats, lons

# Generate pickup coordinates (concentrated in Manhattan business districts)
pickup_lat, pickup_lon = generate_nyc_coordinates(n_samples, 40.75, -73.97, 0.12)

# Generate dropoff coordinates (slightly more dispersed)
dropoff_lat, dropoff_lon = generate_nyc_coordinates(n_samples, 40.73, -73.94, 0.18)

# Generate timestamps (concentrated during rush hours)
base_date = datetime(2023, 1, 1)
hours = np.random.choice(24, n_samples, p=[0.02/1.4, 0.01/1.4, 0.01/1.4, 0.02/1.4, 0.03/1.4, 0.05/1.4, 
                                            0.08/1.4, 0.10/1.4, 0.09/1.4, 0.07/1.4, 0.06/1.4, 0.06/1.4,
                                            0.06/1.4, 0.06/1.4, 0.06/1.4, 0.07/1.4, 0.08/1.4, 0.09/1.4,
                                            0.10/1.4, 0.09/1.4, 0.07/1.4, 0.05/1.4, 0.04/1.4, 0.03/1.4])
minutes = np.random.randint(0, 60, n_samples)
pickup_times = [base_date + pd.Timedelta(hours=int(h), minutes=int(m)) 
                for h, m in zip(hours, minutes)]

# Calculate trip durations (distance-based with traffic factor)
def haversine_distance(lat1, lon1, lat2, lon2):
    """Calculate the great circle distance between two points on earth"""
    lat1, lon1, lat2, lon2 = map(radians, [lat1, lon1, lat2, lon2])
    dlon = lon2 - lon1
    dlat = lat2 - lat1
    a = sin(dlat/2)**2 + cos(lat1) * cos(lat2) * sin(dlon/2)**2
    c = 2 * asin(sqrt(a))
    r = 3956  # Radius of earth in miles
    return c * r

distances = [haversine_distance(p_lat, p_lon, d_lat, d_lon) 
             for p_lat, p_lon, d_lat, d_lon in zip(pickup_lat, pickup_lon, dropoff_lat, dropoff_lon)]

# Trip duration: base time + distance factor + traffic + randomness
traffic_factor = 1 + 0.5 * np.sin(np.array(hours) * np.pi / 12)  # Rush hour traffic
durations = (np.array(distances) * 3 + 5) * traffic_factor + np.random.normal(0, 3, n_samples)
durations = np.clip(durations, 2, 120)  # Realistic bounds

# Generate fare amounts (based on distance and duration)
base_fare = 3.0
per_mile = 2.5
per_minute = 0.5
fares = base_fare + np.array(distances) * per_mile + durations * per_minute
fares = fares + np.random.normal(0, 2, n_samples)  # Add noise
fares = np.clip(fares, 3, 150)

# Passenger count
passenger_counts = np.random.choice([1, 2, 3, 4, 5, 6], n_samples, 
                                   p=[0.60, 0.25, 0.10, 0.03, 0.015, 0.005])

# Payment type
payment_types = np.random.choice(['Credit Card', 'Cash', 'No Charge', 'Dispute', 'Unknown'], 
                                n_samples, p=[0.65, 0.30, 0.03, 0.015, 0.005])

# Create DataFrame
df = pd.DataFrame({
    'pickup_datetime': pickup_times,
    'pickup_latitude': pickup_lat,
    'pickup_longitude': pickup_lon,
    'dropoff_latitude': dropoff_lat,
    'dropoff_longitude': dropoff_lon,
    'trip_distance': distances,
    'trip_duration_minutes': durations,
    'fare_amount': fares,
    'passenger_count': passenger_counts,
    'payment_type': payment_types
})

# Extract temporal features
df['pickup_hour'] = df['pickup_datetime'].dt.hour
df['pickup_day'] = df['pickup_datetime'].dt.day_name()
df['pickup_month'] = df['pickup_datetime'].dt.month

print("✅ NYC Taxi Dataset Created Successfully!")
print(f"📊 Dataset Shape: {df.shape}")
print(f"🚕 Total Trips: {len(df):,}")
print("\n" + "="*50)
df.head(10)

In [ ]:
# Initial data inspection
print("🔍 DATASET OVERVIEW")
print("="*60)
print(f"Shape: {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"\nMemory Usage: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")
print(f"\nDate Range: {df['pickup_datetime'].min()} to {df['pickup_datetime'].max()}")

print("\n📋 COLUMN INFORMATION")
print("="*60)
df.info()

In [ ]:
# Statistical summary
print("📊 STATISTICAL SUMMARY - SPATIAL & TRIP METRICS")
print("="*60)
spatial_cols = ['pickup_latitude', 'pickup_longitude', 'dropoff_latitude', 
                'dropoff_longitude', 'trip_distance', 'trip_duration_minutes', 'fare_amount']
df[spatial_cols].describe().round(3)

### 💡 Key Insight #1: Data Quality Check
Our synthetic dataset mimics real NYC taxi patterns:
- **Pickup concentration**: Higher density in Manhattan (40.7-40.8°N, -74.0 to -73.9°W)
- **Trip distances**: Ranging from very short (0.1 mi) to long trips (20+ mi)
- **Temporal coverage**: Full 24-hour distribution with realistic rush hour peaks

## 3. Understanding Key Spatial Features 📍

Let's examine the coordinate distributions and understand the spatial extent of our data.

In [ ]:
# Analyze coordinate distributions
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('📍 Spatial Feature Distributions', fontsize=16, fontweight='bold')

# Pickup Latitude
axes[0,0].hist(df['pickup_latitude'], bins=50, color='skyblue', edgecolor='black', alpha=0.7)
axes[0,0].set_title('Pickup Latitude Distribution')
axes[0,0].set_xlabel('Latitude')
axes[0,0].set_ylabel('Frequency')
axes[0,0].axvline(df['pickup_latitude'].mean(), color='red', linestyle='--', label=f'Mean: {df["pickup_latitude"].mean():.3f}')
axes[0,0].legend()

# Pickup Longitude
axes[0,1].hist(df['pickup_longitude'], bins=50, color='lightcoral', edgecolor='black', alpha=0.7)
axes[0,1].set_title('Pickup Longitude Distribution')
axes[0,1].set_xlabel('Longitude')
axes[0,1].axvline(df['pickup_longitude'].mean(), color='red', linestyle='--', label=f'Mean: {df["pickup_longitude"].mean():.3f}')
axes[0,1].legend()

# Dropoff Latitude
axes[1,0].hist(df['dropoff_latitude'], bins=50, color='lightgreen', edgecolor='black', alpha=0.7)
axes[1,0].set_title('Dropoff Latitude Distribution')
axes[1,0].set_xlabel('Latitude')
axes[1,0].axvline(df['dropoff_latitude'].mean(), color='red', linestyle='--', label=f'Mean: {df["dropoff_latitude"].mean():.3f}')
axes[1,0].legend()

# Dropoff Longitude
axes[1,1].hist(df['dropoff_longitude'], bins=50, color='gold', edgecolor='black', alpha=0.7)
axes[1,1].set_title('Dropoff Longitude Distribution')
axes[1,1].set_xlabel('Longitude')
axes[1,1].axvline(df['dropoff_longitude'].mean(), color='red', linestyle='--', label=f'Mean: {df["dropoff_longitude"].mean():.3f}')
axes[1,1].legend()

plt.tight_layout()
plt.show()

print("📍 COORDINATE STATISTICS")
print("="*50)
print(f"Pickup Lat Range: [{df['pickup_latitude'].min():.4f}, {df['pickup_latitude'].max():.4f}]")
print(f"Pickup Lon Range: [{df['pickup_longitude'].min():.4f}, {df['pickup_longitude'].max():.4f}]")
print(f"Dropoff Lat Range: [{df['dropoff_latitude'].min():.4f}, {df['dropoff_latitude'].max():.4f}]")
print(f"Dropoff Lon Range: [{df['dropoff_longitude'].min():.4f}, {df['dropoff_longitude'].max():.4f}]")

## 4. Basic Spatial Statistics 🧮

Now let's calculate and analyze fundamental spatial metrics: trip distance, duration, and speed.

In [ ]:
# Calculate derived spatial metrics
df['avg_speed_mph'] = df['trip_distance'] / (df['trip_duration_minutes'] / 60)
df['avg_speed_mph'] = df['avg_speed_mph'].replace([np.inf, -np.inf], np.nan)
df['avg_speed_mph'] = df['avg_speed_mph'].fillna(df['avg_speed_mph'].median())

# Manhattan distance approximation (grid-based city distance)
df['manhattan_distance'] = (abs(df['pickup_latitude'] - df['dropoff_latitude']) + 
                            abs(df['pickup_longitude'] - df['dropoff_longitude'])) * 69  # approx miles

print("🚕 TRIP METRICS SUMMARY")
print("="*60)
metrics = ['trip_distance', 'trip_duration_minutes', 'avg_speed_mph', 'fare_amount']
summary = df[metrics].describe(percentiles=[.25, .5, .75, .90, .95, .99]).round(2)
print(summary)

print("\n💰 FARE BREAKDOWN")
print("="*60)
print(f"Average Fare: ${df['fare_amount'].mean():.2f}")
print(f"Median Fare: ${df['fare_amount'].median():.2f}")
print(f"Fare per Mile: ${(df['fare_amount'] / df['trip_distance']).mean():.2f}")
print(f"Fare per Minute: ${(df['fare_amount'] / df['trip_duration_minutes']).mean():.2f}")

In [ ]:
# Visualize trip metrics distributions
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('🚕 Trip Metrics Analysis', fontsize=16, fontweight='bold')

# Trip Distance
axes[0,0].hist(df['trip_distance'], bins=50, color='steelblue', edgecolor='black', alpha=0.7)
axes[0,0].set_title('Trip Distance Distribution (miles)')
axes[0,0].set_xlabel('Distance (miles)')
axes[0,0].set_ylabel('Frequency')
axes[0,0].axvline(df['trip_distance'].median(), color='red', linestyle='--', linewidth=2, label=f'Median: {df["trip_distance"].median():.2f} mi')
axes[0,0].legend()

# Trip Duration
axes[0,1].hist(df['trip_duration_minutes'], bins=50, color='forestgreen', edgecolor='black', alpha=0.7)
axes[0,1].set_title('Trip Duration Distribution (minutes)')
axes[0,1].set_xlabel('Duration (minutes)')
axes[0,1].axvline(df['trip_duration_minutes'].median(), color='red', linestyle='--', linewidth=2, label=f'Median: {df["trip_duration_minutes"].median():.1f} min')
axes[0,1].legend()

# Average Speed
axes[1,0].hist(df['avg_speed_mph'], bins=50, color='coral', edgecolor='black', alpha=0.7, range=(0, 50))
axes[1,0].set_title('Average Speed Distribution (mph)')
axes[1,0].set_xlabel('Speed (mph)')
axes[1,0].axvline(df['avg_speed_mph'].median(), color='red', linestyle='--', linewidth=2, label=f'Median: {df["avg_speed_mph"].median():.1f} mph')
axes[1,0].legend()

# Fare Amount
axes[1,1].hist(df['fare_amount'], bins=50, color='gold', edgecolor='black', alpha=0.7)
axes[1,1].set_title('Fare Amount Distribution ($)')
axes[1,1].set_xlabel('Fare ($)')
axes[1,1].axvline(df['fare_amount'].median(), color='red', linestyle='--', linewidth=2, label=f'Median: ${df["fare_amount"].median():.2f}')
axes[1,1].legend()

plt.tight_layout()
plt.show()

### 💡 Key Insight #2: Trip Patterns
- **Typical trip**: ~2-3 miles, ~12-15 minutes, ~$12-15 fare
- **Speed patterns**: NYC traffic averages ~10-12 mph (much slower than highway speeds)
- **Outliers**: Some very long trips (>15 miles) likely to airports or outer boroughs

## 5. Visualizing Pickup and Dropoff Locations 📊

Let's create scatter plots and density heatmaps to visualize spatial distributions.

In [ ]:
# Create scatter plot of all pickup and dropoff locations
fig, axes = plt.subplots(1, 2, figsize=(16, 7))
fig.suptitle('🗺️ NYC Taxi Trip Locations', fontsize=16, fontweight='bold')

# Sample for visualization (plotting all 10k points might be dense)
sample_size = 2000
df_sample = df.sample(n=sample_size, random_state=42)

# Pickup locations
axes[0].scatter(df_sample['pickup_longitude'], df_sample['pickup_latitude'], 
                c='blue', alpha=0.4, s=10, label='Pickups')
axes[0].set_title(f'📍 Pickup Locations (n={sample_size:,})')
axes[0].set_xlabel('Longitude')
axes[0].set_ylabel('Latitude')
axes[0].grid(True, alpha=0.3)
axes[0].set_xlim(-74.05, -73.75)
axes[0].set_ylim(40.60, 40.90)

# Dropoff locations
axes[1].scatter(df_sample['dropoff_longitude'], df_sample['dropoff_latitude'], 
                c='red', alpha=0.4, s=10, label='Dropoffs')
axes[1].set_title(f'📍 Dropoff Locations (n={sample_size:,})')
axes[1].set_xlabel('Longitude')
axes[1].set_ylabel('Latitude')
axes[1].grid(True, alpha=0.3)
axes[1].set_xlim(-74.05, -73.75)
axes[1].set_ylim(40.60, 40.90)

plt.tight_layout()
plt.show()

print("🗺️ SPATIAL OBSERVATIONS")
print("="*50)
print("• Pickups concentrated in Manhattan business districts (Midtown, Financial District)")
print("• Dropoffs more dispersed - trips spread to residential areas, airports, outer boroughs")
print("• Clear clustering around major transit hubs and commercial centers")

In [ ]:
# Create 2D density heatmaps using hexbin
fig, axes = plt.subplots(1, 2, figsize=(16, 7))
fig.suptitle('🔥 Trip Density Heatmaps', fontsize=16, fontweight='bold')

# Pickup density
hb1 = axes[0].hexbin(df['pickup_longitude'], df['pickup_latitude'], 
                     gridsize=30, cmap='YlOrRd', mincnt=1)
axes[0].set_title('Pickup Density Heatmap')
axes[0].set_xlabel('Longitude')
axes[0].set_ylabel('Latitude')
axes[0].set_xlim(-74.05, -73.75)
axes[0].set_ylim(40.60, 40.90)
cb1 = plt.colorbar(hb1, ax=axes[0])
cb1.set_label('Trip Count')

# Dropoff density
hb2 = axes[1].hexbin(df['dropoff_longitude'], df['dropoff_latitude'], 
                     gridsize=30, cmap='YlGnBu', mincnt=1)
axes[1].set_title('Dropoff Density Heatmap')
axes[1].set_xlabel('Longitude')
axes[1].set_ylabel('Latitude')
axes[1].set_xlim(-74.05, -73.75)
axes[1].set_ylim(40.60, 40.90)
cb2 = plt.colorbar(hb2, ax=axes[1])
cb2.set_label('Trip Count')

plt.tight_layout()
plt.show()

In [ ]:
# KDE plots for spatial density (seaborn)
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('📊 Spatial Density Estimation (KDE)', fontsize=16, fontweight='bold')

# Sample for KDE (performance)
df_kde = df.sample(n=3000, random_state=42)

# Pickup KDE
sns.kdeplot(data=df_kde, x='pickup_longitude', y='pickup_latitude', 
            cmap='Reds', fill=True, thresh=0.05, ax=axes[0])
axes[0].set_title('Pickup Location Density')
axes[0].set_xlim(-74.05, -73.75)
axes[0].set_ylim(40.60, 40.90)

# Dropoff KDE
sns.kdeplot(data=df_kde, x='dropoff_longitude', y='dropoff_latitude', 
            cmap='Blues', fill=True, thresh=0.05, ax=axes[1])
axes[1].set_title('Dropoff Location Density')
axes[1].set_xlim(-74.05, -73.75)
axes[1].set_ylim(40.60, 40.90)

plt.tight_layout()
plt.show()

### 💡 Key Insight #3: Spatial Hotspots
The heatmaps reveal:
- **Pickup hotspots**: Midtown Manhattan (40.75°N, -73.98°W), Financial District, major hotels
- **Dropoff dispersion**: More spread out, indicating trips from commercial to residential areas
- **Airport corridors**: Some linear patterns suggesting JFK/LGA airport routes

## 6. Interactive Maps with Plotly 🌐

Now let's create stunning interactive visualizations using Plotly!

In [ ]:
# Create interactive scatter map for pickups
# Sample for performance
df_map = df.sample(n=1500, random_state=42).copy()
df_map['size'] = df_map['trip_distance'] * 3  # Size by trip distance

fig = px.scatter_mapbox(df_map, 
                        lat='pickup_latitude', 
                        lon='pickup_longitude',
                        color='fare_amount',
                        size='size',
                        hover_data=['trip_distance', 'trip_duration_minutes', 'pickup_hour'],
                        color_continuous_scale='Viridis',
                        title='🚕 Interactive NYC Taxi Pickup Map (Colored by Fare)',
                        zoom=11,
                        height=600)

fig.update_layout(mapbox_style='carto-darkmatter')
fig.update_layout(margin={'r':0,'t':50,'l':0,'b':0})
fig.show()

print("🗺️ INTERACTIVE MAP FEATURES:")
print("• Zoom and pan to explore different areas")
print("• Hover over points for trip details")
print("• Color indicates fare amount (darker = higher fare)")
print("• Size indicates trip distance")

In [ ]:
# Create a combined pickup and dropoff comparison map
df_viz = df.sample(n=1000, random_state=42)

# Create pickup points
pickup_trace = go.Scattermapbox(
    lat=df_viz['pickup_latitude'],
    lon=df_viz['pickup_longitude'],
    mode='markers',
    marker=dict(size=8, color='blue', opacity=0.6),
    name='Pickups',
    text=[f"Pickup<br>Fare: ${f:.2f}<br>Dist: {d:.1f} mi" 
          for f, d in zip(df_viz['fare_amount'], df_viz['trip_distance'])],
    hoverinfo='text'
)

# Create dropoff points
dropoff_trace = go.Scattermapbox(
    lat=df_viz['dropoff_latitude'],
    lon=df_viz['dropoff_longitude'],
    mode='markers',
    marker=dict(size=8, color='red', opacity=0.6),
    name='Dropoffs',
    text=[f"Dropoff<br>Duration: {d:.1f} min<br>Speed: {s:.1f} mph" 
          for d, s in zip(df_viz['trip_duration_minutes'], df_viz['avg_speed_mph'])],
    hoverinfo='text'
)

layout = go.Layout(
    title='🗺️ NYC Taxi: Pickups (Blue) vs Dropoffs (Red)',
    mapbox=dict(
        style='open-street-map',
        center=dict(lat=40.75, lon=-73.95),
        zoom=11
    ),
    height=600,
    showlegend=True
)

fig = go.Figure(data=[pickup_trace, dropoff_trace], layout=layout)
fig.show()

In [ ]:
# Create a density map (heatmap) using Plotly
fig = px.density_mapbox(df.sample(n=3000, random_state=42), 
                        lat='pickup_latitude', 
                        lon='pickup_longitude',
                        z='fare_amount',
                        radius=10,
                        center=dict(lat=40.75, lon=-73.95),
                        zoom=11,
                        mapbox_style='stamen-terrain',
                        title='🔥 Pickup Density Heatmap (Weighted by Fare)',
                        height=600,
                        color_continuous_scale='Hot')

fig.update_layout(margin={'r':0,'t':50,'l':0,'b':0})
fig.show()

### 💡 Key Insight #4: Interactive Exploration Benefits
Interactive maps allow us to:
- **Identify exact hotspot coordinates** by zooming
- **Correlate location with fare/distance** via hover tooltips
- **Discover spatial outliers** (trips to unusual locations)
- **Compare pickup vs dropoff patterns** side-by-side

## 7. Analyzing Popular Pickup/Dropoff Zones 🎯

Let's identify and analyze the most popular zones using coordinate clustering.

In [ ]:
# Create grid-based zones (simplified borough/neighborhood approximation)
# Define coordinate bins for zone classification
def assign_zone(lat, lon):
    """Simple zone assignment based on coordinates"""
    if lat > 40.80 and lon < -73.90:
        return 'Bronx/North'
    elif lat > 40.75 and lon > -73.95 and lon < -73.85:
        return 'Manhattan East'
    elif lat > 40.75 and lon <= -73.95:
        return 'Manhattan West'
    elif lat > 40.70 and lat <= 40.75:
        return 'Midtown'
    elif lat <= 40.70 and lon > -73.95:
        return 'Brooklyn/Queens'
    else:
        return 'Other'

df['pickup_zone'] = df.apply(lambda x: assign_zone(x['pickup_latitude'], x['pickup_longitude']), axis=1)
df['dropoff_zone'] = df.apply(lambda x: assign_zone(x['dropoff_latitude'], x['dropoff_longitude']), axis=1)

# Analyze popular zones
print("🏙️ PICKUP ZONE ANALYSIS")
print("="*60)
pickup_zones = df['pickup_zone'].value_counts()
print(pickup_zones)
print(f"\nTop zone represents {pickup_zones.iloc[0]/len(df)*100:.1f}% of all pickups")

print("\n🏙️ DROPOFF ZONE ANALYSIS")
print("="*60)
dropoff_zones = df['dropoff_zone'].value_counts()
print(dropoff_zones)

In [ ]:
# Visualize zone popularity
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle('🏙️ Trip Volume by Zone', fontsize=16, fontweight='bold')

# Pickup zones
pickup_zones.plot(kind='bar', ax=axes[0], color='skyblue', edgecolor='black')
axes[0].set_title('Pickup Volume by Zone')
axes[0].set_xlabel('Zone')
axes[0].set_ylabel('Number of Trips')
axes[0].tick_params(axis='x', rotation=45)

# Dropoff zones
dropoff_zones.plot(kind='bar', ax=axes[1], color='lightcoral', edgecolor='black')
axes[1].set_title('Dropoff Volume by Zone')
axes[1].set_xlabel('Zone')
axes[1].set_ylabel('Number of Trips')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

In [ ]:
# Analyze zone-to-zone flow (OD matrix sample)
zone_flow = pd.crosstab(df['pickup_zone'], df['dropoff_zone'], margins=True)
print("🔄 ZONE-TO-ZONE FLOW MATRIX (Origin-Destination)")
print("="*70)
print("Rows = Pickup Zone, Columns = Dropoff Zone")
print(zone_flow)

# Calculate flow percentages
zone_flow_pct = pd.crosstab(df['pickup_zone'], df['dropoff_zone'], normalize='index') * 100
print("\n📊 FLOW PERCENTAGES (% of trips from each pickup zone)")
print("="*70)
print(zone_flow_pct.round(1))

In [ ]:
# Visualize zone flow as heatmap
plt.figure(figsize=(10, 8))
sns.heatmap(zone_flow.iloc[:-1, :-1], annot=True, fmt='d', cmap='YlOrRd', 
            cbar_kws={'label': 'Number of Trips'})
plt.title('🔄 Zone-to-Zone Trip Flow Heatmap', fontsize=14, fontweight='bold')
plt.xlabel('Dropoff Zone', fontsize=12)
plt.ylabel('Pickup Zone', fontsize=12)
plt.tight_layout()
plt.show()

print("💡 INSIGHT: The diagonal shows intra-zone trips (same zone pickup/dropoff)")
print("💡 Off-diagonal shows inter-zone travel patterns")

### 💡 Key Insight #5: Zone Patterns
- **Manhattan dominance**: Most trips originate in Manhattan zones
- **Intra-zone preference**: ~40-50% of trips stay within the same zone
- **Brooklyn/Queens inbound**: Significant flow from outer boroughs to Manhattan
- **Asymmetric flows**: Different patterns for morning vs evening (rush hour effects)

## 8. Trip Distance and Fare Analysis by Location 💰

Let's analyze how distance and fare vary by location zones.

In [ ]:
# Distance analysis by zone
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('📊 Trip Metrics by Zone', fontsize=16, fontweight='bold')

# Boxplot: Distance by Pickup Zone
df.boxplot(column='trip_distance', by='pickup_zone', ax=axes[0,0])
axes[0,0].set_title('Trip Distance by Pickup Zone')
axes[0,0].set_xlabel('Pickup Zone')
axes[0,0].set_ylabel('Distance (miles)')
axes[0,0].tick_params(axis='x', rotation=45)

# Boxplot: Fare by Pickup Zone
df.boxplot(column='fare_amount', by='pickup_zone', ax=axes[0,1])
axes[0,1].set_title('Fare Amount by Pickup Zone')
axes[0,1].set_xlabel('Pickup Zone')
axes[0,1].set_ylabel('Fare ($)')
axes[0,1].tick_params(axis='x', rotation=45)

# Average metrics by zone
zone_stats = df.groupby('pickup_zone')[['trip_distance', 'fare_amount', 'trip_duration_minutes']].mean()

zone_stats['trip_distance'].plot(kind='bar', ax=axes[1,0], color='steelblue', edgecolor='black')
axes[1,0].set_title('Average Distance by Zone')
axes[1,0].set_ylabel('Miles')
axes[1,0].tick_params(axis='x', rotation=45)

zone_stats['fare_amount'].plot(kind='bar', ax=axes[1,1], color='gold', edgecolor='black')
axes[1,1].set_title('Average Fare by Zone')
axes[1,1].set_ylabel('Dollars ($)')
axes[1,1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

print("📈 ZONE STATISTICS SUMMARY")
print("="*60)
print(zone_stats.round(2))

In [ ]:
# Correlation analysis: Distance vs Fare by Zone
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle('💰 Distance vs Fare Relationship', fontsize=16, fontweight='bold')

# Scatter plot colored by zone
for zone in df['pickup_zone'].unique():
    zone_data = df[df['pickup_zone'] == zone].sample(min(500, len(df[df['pickup_zone'] == zone])))
    axes[0].scatter(zone_data['trip_distance'], zone_data['fare_amount'], 
                   label=zone, alpha=0.5, s=20)

axes[0].set_xlabel('Trip Distance (miles)')
axes[0].set_ylabel('Fare Amount ($)')
axes[0].set_title('Distance vs Fare by Pickup Zone')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Regression plot overall
sns.regplot(data=df.sample(2000), x='trip_distance', y='fare_amount', 
           scatter_kws={'alpha':0.3}, line_kws={'color':'red'}, ax=axes[1])
axes[1].set_title('Distance-Fare Correlation (with Regression Line)')
axes[1].set_xlabel('Trip Distance (miles)')
axes[1].set_ylabel('Fare Amount ($)')

plt.tight_layout()
plt.show()

# Calculate correlation
corr = df['trip_distance'].corr(df['fare_amount'])
print(f"📊 Distance-Fare Correlation: {corr:.3f}")
print(f"💡 Interpretation: {'Strong' if abs(corr) > 0.7 else 'Moderate' if abs(corr) > 0.4 else 'Weak'} positive relationship")

In [ ]:
# Fare efficiency analysis (fare per mile)
df['fare_per_mile'] = df['fare_amount'] / df['trip_distance']
df['fare_per_mile'] = df['fare_per_mile'].replace([np.inf, -np.inf], np.nan)
df['fare_per_mile'] = df['fare_per_mile'].fillna(df['fare_per_mile'].median())

plt.figure(figsize=(10, 6))
df.boxplot(column='fare_per_mile', by='pickup_zone')
plt.title('💰 Fare Efficiency by Zone ($/mile)')
plt.xlabel('Pickup Zone')
plt.ylabel('Fare per Mile ($)')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

print("🚕 FARE EFFICIENCY ANALYSIS")
print("="*60)
efficiency_stats = df.groupby('pickup_zone')['fare_per_mile'].agg(['mean', 'median', 'std']).round(2)
print(efficiency_stats)
print("\n💡 Note: Higher $/mile suggests more traffic (slower trips) or airport surcharges")

### 💡 Key Insight #6: Location-Based Economics
- **Distance-Fare correlation**: Strong positive (~0.85), but with variance due to traffic and surcharges
- **Zone premiums**: Manhattan trips command higher fares per mile due to congestion
- **Airport effect**: Some zones show higher variance (likely including airport trips with flat rates)
- **Efficiency varies**: Short trips in traffic have higher $/mile ratios

## 9. Temporal + Spatial Patterns ⏰🗺️

Combining time and location reveals powerful insights about urban mobility.

In [ ]:
# Analyze pickup patterns by hour of day
hourly_pickups = df.groupby(['pickup_hour', 'pickup_zone']).size().unstack(fill_value=0)

plt.figure(figsize=(12, 6))
hourly_pickups.plot(kind='line', marker='o', linewidth=2, markersize=4)
plt.title('🕐 Hourly Pickup Patterns by Zone', fontsize=14, fontweight='bold')
plt.xlabel('Hour of Day')
plt.ylabel('Number of Pickups')
plt.legend(title='Zone', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.grid(True, alpha=0.3)
plt.xticks(range(0, 24, 2))
plt.tight_layout()
plt.show()

print("⏰ PEAK HOURS BY ZONE")
print("="*60)
for zone in hourly_pickups.columns:
    peak_hour = hourly_pickups[zone].idxmax()
    peak_count = hourly_pickups[zone].max()
    print(f"{zone:20s}: Peak at {peak_hour:2d}:00 ({peak_count} pickups)")

In [ ]:
# Rush hour spatial analysis
df['time_category'] = df['pickup_hour'].apply(
    lambda x: 'Morning Rush (7-9)' if 7 <= x <= 9 
    else 'Evening Rush (17-19)' if 17 <= x <= 19
    else 'Midday (10-16)' if 10 <= x <= 16
    else 'Night (20-6)'
)

# Spatial distribution by time category
fig, axes = plt.subplots(2, 2, figsize=(14, 12))
fig.suptitle('🕐 Spatial Patterns by Time of Day', fontsize=16, fontweight='bold')

time_categories = ['Morning Rush (7-9)', 'Midday (10-16)', 'Evening Rush (17-19)', 'Night (20-6)']
colors = ['red', 'orange', 'blue', 'purple']

for idx, (time_cat, color) in enumerate(zip(time_categories, colors)):
    ax = axes[idx // 2, idx % 2]
    data = df[df['time_category'] == time_cat].sample(min(500, len(df[df['time_category'] == time_cat])))
    
    ax.scatter(data['pickup_longitude'], data['pickup_latitude'], 
              c=color, alpha=0.4, s=15, label=f'{time_cat} (n={len(data)})')
    ax.set_title(f'{time_cat}')
    ax.set_xlabel('Longitude')
    ax.set_ylabel('Latitude')
    ax.set_xlim(-74.05, -73.75)
    ax.set_ylim(40.60, 40.90)
    ax.grid(True, alpha=0.3)
    ax.legend()

plt.tight_layout()
plt.show()

In [ ]:
# Average trip metrics by hour
hourly_metrics = df.groupby('pickup_hour')[['trip_distance', 'trip_duration_minutes', 
                                           'avg_speed_mph', 'fare_amount']].mean()

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('📊 Hourly Trip Metrics (Temporal Patterns)', fontsize=16, fontweight='bold')

hourly_metrics['trip_distance'].plot(ax=axes[0,0], marker='o', color='steelblue')
axes[0,0].set_title('Average Trip Distance by Hour')
axes[0,0].set_ylabel('Miles')
axes[0,0].grid(True, alpha=0.3)

hourly_metrics['trip_duration_minutes'].plot(ax=axes[0,1], marker='o', color='forestgreen')
axes[0,1].set_title('Average Trip Duration by Hour')
axes[0,1].set_ylabel('Minutes')
axes[0,1].grid(True, alpha=0.3)

hourly_metrics['avg_speed_mph'].plot(ax=axes[1,0], marker='o', color='coral')
axes[1,0].set_title('Average Speed by Hour')
axes[1,0].set_ylabel('MPH')
axes[1,0].grid(True, alpha=0.3)

hourly_metrics['fare_amount'].plot(ax=axes[1,1], marker='o', color='gold')
axes[1,1].set_title('Average Fare by Hour')
axes[1,1].set_ylabel('Dollars ($)')
axes[1,1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("⏰ TEMPORAL INSIGHTS")
print("="*60)
print(f"Slowest speed: {hourly_metrics['avg_speed_mph'].min():.1f} mph at {hourly_metrics['avg_speed_mph'].idxmin()}:00")
print(f"Fastest speed: {hourly_metrics['avg_speed_mph'].max():.1f} mph at {hourly_metrics['avg_speed_mph'].idxmax()}:00")
print(f"Longest trips: {hourly_metrics['trip_distance'].max():.1f} miles at {hourly_metrics['trip_distance'].idxmax()}:00")
print(f"Highest fares: ${hourly_metrics['fare_amount'].max():.2f} at {hourly_metrics['fare_amount'].idxmax()}:00")

### 💡 Key Insight #7: Rush Hour Dynamics
- **Morning rush**: Concentrated pickups in residential areas (Brooklyn/Queens → Manhattan)
- **Evening rush**: Dispersed dropoffs as people return home
- **Speed patterns**: Traffic significantly slows during rush hours (8-10 mph vs 15+ mph at night)
- **Distance patterns**: Longer trips during off-peak hours (airport runs, cross-borough travel)

## 10. Key Insights for Geospatial Modeling and Applications 🤖

Let's summarize the key features and patterns useful for machine learning models.

In [ ]:
# Feature engineering summary for ML
print("🤖 GEOSPATIAL FEATURES FOR MACHINE LEARNING")
print("="*70)
print("\n1️⃣  LOCATION-BASED FEATURES:")
print("   • pickup_latitude, pickup_longitude (raw coordinates)")
print("   • dropoff_latitude, dropoff_longitude (raw coordinates)")
print("   • pickup_zone, dropoff_zone (categorical zones)")
print("   • haversine_distance (great circle distance)")
print("   • manhattan_distance (grid-based approximation)")

print("\n2️⃣  TEMPORAL-SPATIAL FEATURES:")
print("   • pickup_hour (0-23)")
print("   • pickup_day (day of week)")
print("   • time_category (rush hour classification)")
print("   • is_weekend (boolean)")
print("   • is_rush_hour (boolean)")

print("\n3️⃣  DERIVED SPATIAL FEATURES:")
print("   • direction (bearing from pickup to dropoff)")
print("   • centroid_distance (distance from city center)")
print("   • zone_transition (pickup_zone → dropoff_zone)")
print("   • airport_indicator (likely airport trip)")

print("\n4️⃣  AGGREGATION FEATURES (require historical data):")
print("   • avg_pickups_in_zone_last_hour")
print("   • avg_trips_from_zone_at_this_hour")
print("   • typical_fare_for_this_od_pair")

# Create additional features
df['is_weekend'] = df['pickup_day'].isin(['Saturday', 'Sunday'])
df['is_rush_hour'] = df['pickup_hour'].isin([7, 8, 9, 17, 18, 19])

# Calculate bearing (direction)
def calculate_bearing(lat1, lon1, lat2, lon2):
    """Calculate bearing between two points"""
    lat1, lon1, lat2, lon2 = map(radians, [lat1, lon1, lat2, lon2])
    dlon = lon2 - lon1
    x = sin(dlon) * cos(lat2)
    y = cos(lat1) * sin(lat2) - sin(lat1) * cos(lat2) * cos(dlon)
    bearing = (np.degrees(np.arctan2(x, y)) + 360) % 360
    return bearing

df['bearing'] = df.apply(lambda x: calculate_bearing(x['pickup_latitude'], x['pickup_longitude'],
                                                       x['dropoff_latitude'], x['dropoff_longitude']), axis=1)

# Distance from city center (Times Square approx: 40.758, -73.985)
city_center_lat, city_center_lon = 40.758, -73.985
df['pickup_dist_from_center'] = df.apply(lambda x: haversine_distance(x['pickup_latitude'], x['pickup_longitude'],
                                                                        city_center_lat, city_center_lon), axis=1)

print("\n✅ FEATURE ENGINEERING COMPLETE")
print(f"📊 Total features available: {df.shape[1]}")
print("\n🎯 SAMPLE OF ENGINEERED FEATURES:")
feature_cols = ['trip_distance', 'pickup_hour', 'is_weekend', 'is_rush_hour', 
               'bearing', 'pickup_dist_from_center', 'fare_per_mile']
print(df[feature_cols].head().round(2))

In [ ]:
# Model application examples
print("🚀 REAL-WORLD ML APPLICATIONS")
print("="*70)

applications = {
    "Demand Prediction": {
        "Target": "Number of pickups in next hour per zone",
        "Features": ["hour", "day", "weather", "historical_demand", "events"],
        "Use Case": "Pre-position vehicles in high-demand zones"
    },
    "Fare Estimation": {
        "Target": "Trip fare amount",
        "Features": ["pickup_loc", "dropoff_loc", "hour", "distance", "traffic"],
        "Use Case": "Upfront price estimates for passengers"
    },
    "Trip Duration Prediction": {
        "Target": "ETA / trip duration",
        "Features": ["route", "hour", "day", "weather", "traffic_conditions"],
        "Use Case": "Accurate arrival time predictions"
    },
    "Anomaly Detection": {
        "Target": "Unusual trip patterns",
        "Features": ["distance", "duration", "fare", "locations", "time"],
        "Use Case": "Fraud detection, driver/passenger safety"
    },
    "Route Optimization": {
        "Target": "Optimal path between points",
        "Features": ["origin", "destination", "time", "traffic", "road_network"],
        "Use Case": "Minimize travel time and fuel consumption"
    }
}

for app, details in applications.items():
    print(f"\n📱 {app}")
    print(f"   Target: {details['Target']}")
    print(f"   Key Features: {', '.join(details['Features'])}")
    print(f"   Use Case: {details['Use Case']}")

print("\n" + "="*70)
print("💡 KEY MODELING INSIGHTS:")
print("   • Spatial autocorrelation: Nearby locations have similar patterns")
print("   • Temporal seasonality: Strong daily/weekly cycles")
print("   • Non-linear relationships: Distance vs fare, time vs speed")
print("   • Categorical importance: Zones, hours, days are strong predictors")
print("   • Feature interactions: Zone × Hour combinations crucial for demand")

## 🛠️ Hands-On Exercises

Now it's your turn to practice geospatial analysis! Complete these exercises to reinforce your learning.

### Exercise 1: Visualize Pickup Density by Hour
Create separate density plots for pickups during morning rush (7-9 AM) vs evening rush (5-7 PM). Compare the spatial distributions.

### Exercise 2: Create Interactive Time-Based Map
Build an interactive Plotly map that colors trips by hour of day, allowing you to see how pickup patterns change throughout the day.

### Exercise 3: Analyze Trip Distance by Zone Pairs
Calculate and visualize the average trip distance for each pickup zone → dropoff zone combination. Which zone pairs have the longest trips?

### Exercise 4: Discover Spatial Outliers
Identify trips with unusually long distances (>99th percentile) or unusual pickup/dropoff locations (outside NYC bounds). Visualize these outliers on a map.

### Exercise 5: Rush Hour Pattern Analysis
Compare average speed, distance, and fare between morning rush (7-9 AM) and evening rush (5-7 PM). Are there significant differences?

### Exercise 6: Geospatial Feature Engineering
Create 3 new spatial features: (1) Direction category (N, S, E, W, NE, etc.), (2) Distance from JFK Airport, (3) Whether trip is within Manhattan only.

### Exercise 7: Airport Trip Analysis
Identify likely airport trips (long distance, specific directions, high fare). Analyze their characteristics: time of day, day of week, typical fare.

### Exercise 8: Build a Geospatial EDA Summary Report
Create a comprehensive summary report including: total trips, coverage area, top 5 pickup zones, peak hours, average metrics, and key spatial insights.

### Exercise 9: Predictive Feature Analysis
Which spatial and temporal features would be most predictive of (a) high fare, (b) long duration, (c) airport trip? Justify your choices with visualizations.

### Exercise 10: Custom Zone Definition
Define your own zones using coordinate clustering (KMeans) or grid-based approach. Compare your custom zones with the simple zones we used. Which is better for analysis?

## Solutions & Key Insights (Review After Attempting) 🔑

### Exercise 1 Solution: Pickup Density by Hour

In [ ]:
# Solution 1: Compare morning vs evening rush density
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

morning_data = df[(df['pickup_hour'] >= 7) & (df['pickup_hour'] <= 9)]
evening_data = df[(df['pickup_hour'] >= 17) & (df['pickup_hour'] <= 19)]

# Morning rush
axes[0].hexbin(morning_data['pickup_longitude'], morning_data['pickup_latitude'], 
               gridsize=25, cmap='Reds', mincnt=1)
axes[0].set_title('Morning Rush (7-9 AM) Pickup Density')
axes[0].set_xlim(-74.05, -73.75)
axes[0].set_ylim(40.60, 40.90)

# Evening rush
axes[1].hexbin(evening_data['pickup_longitude'], evening_data['pickup_latitude'], 
               gridsize=25, cmap='Blues', mincnt=1)
axes[1].set_title('Evening Rush (5-7 PM) Pickup Density')
axes[1].set_xlim(-74.05, -73.75)
axes[1].set_ylim(40.60, 40.90)

plt.tight_layout()
plt.show()

print("💡 INSIGHT: Morning pickups concentrated in residential/outer boroughs;")
print("   Evening pickups more dispersed across business districts")

### Exercise 2 Solution: Interactive Time-Based Map

In [ ]:
# Solution 2: Interactive map with hour-based coloring
df_sample = df.sample(n=2000, random_state=42)

fig = px.scatter_mapbox(df_sample, 
                        lat='pickup_latitude', 
                        lon='pickup_longitude',
                        color='pickup_hour',
                        size='trip_distance',
                        hover_data=['pickup_zone', 'fare_amount'],
                        color_continuous_scale='Turbo',
                        title='🕐 Interactive Map: Pickups Colored by Hour of Day',
                        zoom=11,
                        height=600)

fig.update_layout(mapbox_style='carto-positron')
fig.update_layout(margin={'r':0,'t':50,'l':0,'b':0})
fig.show()

print("💡 INSIGHT: Use the color scale to see temporal patterns:")
print("   Cool colors (blue/purple) = night/early morning")
print("   Warm colors (yellow/red) = afternoon/evening")

### Exercise 3 Solution: Zone Pair Distance Analysis

In [ ]:
# Solution 3: Zone pair distance analysis
zone_pair_dist = df.groupby(['pickup_zone', 'dropoff_zone'])['trip_distance'].agg(['mean', 'count']).round(2)
zone_pair_dist = zone_pair_dist[zone_pair_dist['count'] >= 50]  # Filter for significance
zone_pair_dist = zone_pair_dist.sort_values('mean', ascending=False)

print("🚕 TOP 10 LONGEST AVERAGE DISTANCE ZONE PAIRS:")
print("="*60)
print(zone_pair_dist.head(10))

# Visualization
plt.figure(figsize=(12, 6))
top_pairs = zone_pair_dist.head(10)
pair_labels = [f"{idx[0]} → {idx[1]}" for idx in top_pairs.index]
plt.barh(range(len(top_pairs)), top_pairs['mean'], color='steelblue')
plt.yticks(range(len(top_pairs)), pair_labels)
plt.xlabel('Average Distance (miles)')
plt.title('Top 10 Zone Pairs by Average Trip Distance')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

print("\n💡 INSIGHT: Longest trips typically involve outer boroughs → Manhattan or airport routes")

### Exercise 4 Solution: Spatial Outliers

In [ ]:
# Solution 4: Identify and visualize outliers
distance_threshold = df['trip_distance'].quantile(0.99)
outliers = df[df['trip_distance'] > distance_threshold]

# Also check coordinate outliers
lat_min, lat_max = 40.48, 40.92
lon_min, lon_max = -74.30, -73.70
coord_outliers = df[(df['pickup_latitude'] < lat_min) | (df['pickup_latitude'] > lat_max) |
                   (df['pickup_longitude'] < lon_min) | (df['pickup_longitude'] > lon_max)]

print(f"🚨 DISTANCE OUTLIERS (>99th percentile, {distance_threshold:.1f} miles): {len(outliers)} trips")
print(f"🚨 COORDINATE OUTLIERS (outside NYC): {len(coord_outliers)} trips")

# Visualize
fig, ax = plt.subplots(figsize=(10, 8))
ax.scatter(df['pickup_longitude'], df['pickup_latitude'], c='lightblue', alpha=0.3, s=10, label='Normal')
ax.scatter(outliers['pickup_longitude'], outliers['pickup_latitude'], 
          c='red', s=50, marker='x', label=f'Distance Outliers (n={len(outliers)})')
if len(coord_outliers) > 0:
    ax.scatter(coord_outliers['pickup_longitude'], coord_outliers['pickup_latitude'], 
              c='orange', s=50, marker='^', label=f'Coord Outliers (n={len(coord_outliers)})')

ax.set_xlim(-74.05, -73.75)
ax.set_ylim(40.60, 40.90)
ax.set_title('Spatial Outliers Detection')
ax.legend()
ax.grid(True, alpha=0.3)
plt.show()

print("\n💡 INSIGHT: Outliers may indicate:")
print("   • Airport trips (JFK ~18 miles, LGA ~8 miles from Manhattan)")
print("   • Data entry errors (coordinate outliers)")
print("   • Special events or unusual travel patterns")

### Exercise 5 Solution: Rush Hour Comparison

In [ ]:
# Solution 5: Rush hour comparison
morning_rush = df[(df['pickup_hour'] >= 7) & (df['pickup_hour'] <= 9)]
evening_rush = df[(df['pickup_hour'] >= 17) & (df['pickup_hour'] <= 19)]

comparison = pd.DataFrame({
    'Morning Rush (7-9)': [
        morning_rush['avg_speed_mph'].mean(),
        morning_rush['trip_distance'].mean(),
        morning_rush['fare_amount'].mean(),
        morning_rush['trip_duration_minutes'].mean()
    ],
    'Evening Rush (17-19)': [
        evening_rush['avg_speed_mph'].mean(),
        evening_rush['trip_distance'].mean(),
        evening_rush['fare_amount'].mean(),
        evening_rush['trip_duration_minutes'].mean()
    ]
}, index=['Avg Speed (mph)', 'Avg Distance (mi)', 'Avg Fare ($)', 'Avg Duration (min)'])

print("⏰ RUSH HOUR COMPARISON")
print("="*60)
print(comparison.round(2))

# Statistical significance test
from scipy import stats
speed_ttest = stats.ttest_ind(morning_rush['avg_speed_mph'], evening_rush['avg_speed_mph'])
print(f"\n📊 Speed difference t-test p-value: {speed_ttest.pvalue:.4f}")
print(f"💡 {'Statistically significant' if speed_ttest.pvalue < 0.05 else 'Not significant'} difference")

# Visualization
comparison.T.plot(kind='bar', figsize=(12, 6), subplots=True, layout=(2,2), 
                 legend=False, color=['steelblue', 'coral'])
plt.suptitle('Rush Hour Metrics Comparison', fontsize=14)
plt.tight_layout()
plt.show()

### Exercise 6 Solution: Custom Features

In [ ]:
# Solution 6: Advanced feature engineering

# 1. Direction category
def get_direction_category(bearing):
    if 0 <= bearing < 45 or 315 <= bearing <= 360:
        return 'N'
    elif 45 <= bearing < 135:
        return 'E'
    elif 135 <= bearing < 225:
        return 'S'
    else:
        return 'W'

df['direction_category'] = df['bearing'].apply(get_direction_category)

# 2. Distance from JFK Airport (40.6413° N, 73.7781° W)
jfk_lat, jfk_lon = 40.6413, -73.7781
df['dist_from_jfk'] = df.apply(lambda x: haversine_distance(x['pickup_latitude'], x['pickup_longitude'],
                                                            jfk_lat, jfk_lon), axis=1)

# 3. Manhattan-only indicator
manhattan_bounds = {
    'lat_min': 40.70, 'lat_max': 40.88,
    'lon_min': -74.02, 'lon_max': -73.93
}
df['is_manhattan_trip'] = (
    (df['pickup_latitude'].between(manhattan_bounds['lat_min'], manhattan_bounds['lat_max'])) &
    (df['pickup_longitude'].between(manhattan_bounds['lon_min'], manhattan_bounds['lon_max'])) &
    (df['dropoff_latitude'].between(manhattan_bounds['lat_min'], manhattan_bounds['lat_max'])) &
    (df['dropoff_longitude'].between(manhattan_bounds['lon_min'], manhattan_bounds['lon_max']))
)

print("✅ NEW FEATURES CREATED:")
print("="*60)
print(f"Direction distribution:\n{df['direction_category'].value_counts()}")
print(f"\nManhattan-only trips: {df['is_manhattan_trip'].sum():,} ({df['is_manhattan_trip'].mean()*100:.1f}%)")
print(f"Average distance from JFK: {df['dist_from_jfk'].mean():.2f} miles")

# Visualize direction impact
plt.figure(figsize=(10, 6))
df.groupby('direction_category')['trip_distance'].mean().plot(kind='bar', color='teal')
plt.title('Average Trip Distance by Direction')
plt.ylabel('Miles')
plt.xticks(rotation=0)
plt.grid(True, alpha=0.3)
plt.show()

### Exercise 7 Solution: Airport Trip Detection

In [ ]:
# Solution 7: Airport trip analysis
# Define criteria for airport trips
jfk_threshold = 15  # miles from JFK
lga_lat, lga_lon = 40.7769, -73.8740
df['dist_from_lga'] = df.apply(lambda x: haversine_distance(x['pickup_latitude'], x['pickup_longitude'],
                                                            lga_lat, lga_lon), axis=1)

# Airport trip indicators
df['is_airport_pickup'] = (df['dist_from_jfk'] < 3) | (df['dist_from_lga'] < 3)
df['is_airport_dropoff'] = ((df['trip_distance'] > 15) & (df['fare_amount'] > 50) & 
                           (df['direction_category'] == 'E'))  # Rough heuristic

airport_trips = df[df['is_airport_pickup'] | df['is_airport_dropoff']]

print(f"✈️ IDENTIFIED AIRPORT TRIPS: {len(airport_trips)} ({len(airport_trips)/len(df)*100:.1f}%)")
print("\n📊 AIRPORT TRIP CHARACTERISTICS:")
print("="*60)
print(airport_trips[['trip_distance', 'trip_duration_minutes', 'fare_amount', 'pickup_hour']].describe().round(2))

# Time pattern
plt.figure(figsize=(10, 6))
airport_trips['pickup_hour'].value_counts().sort_index().plot(kind='bar', color='navy')
plt.title('Airport Trips by Hour of Day')
plt.xlabel('Hour')
plt.ylabel('Number of Trips')
plt.grid(True, alpha=0.3)
plt.show()

print("\n💡 INSIGHT: Airport trips show distinct patterns:")
print(f"   • Average distance: {airport_trips['trip_distance'].mean():.1f} miles")
print(f"   • Average fare: ${airport_trips['fare_amount'].mean():.2f}")
print(f"   • Peak hours: {airport_trips['pickup_hour'].value_counts().head(3).index.tolist()}")

### Exercise 8 Solution: EDA Summary Report

In [ ]:
# Solution 8: Comprehensive EDA report
def generate_eda_report(df):
    report = []
    report.append("="*70)
    report.append("NYC TAXI GEOSPATIAL EDA SUMMARY REPORT")
    report.append("="*70)
    
    # Basic stats
    report.append(f"\n📊 DATASET OVERVIEW")
    report.append(f"   Total Trips: {len(df):,}")
    report.append(f"   Date Range: {df['pickup_datetime'].min()} to {df['pickup_datetime'].max()}")
    report.append(f"   Coverage Area: {df['pickup_latitude'].min():.3f}°N to {df['pickup_latitude'].max():.3f}°N")
    report.append(f"                   {df['pickup_longitude'].min():.3f}°W to {df['pickup_longitude'].max():.3f}°W")
    
    # Top zones
    report.append(f"\n🏙️ TOP 5 PICKUP ZONES")
    for i, (zone, count) in enumerate(df['pickup_zone'].value_counts().head(5).items(), 1):
        report.append(f"   {i}. {zone}: {count:,} trips ({count/len(df)*100:.1f}%)")
    
    # Peak hours
    report.append(f"\n⏰ PEAK HOURS")
    peak_hours = df['pickup_hour'].value_counts().head(3)
    for hour, count in peak_hours.items():
        report.append(f"   {hour}:00 - {count:,} trips")
    
    # Average metrics
    report.append(f"\n📈 AVERAGE METRICS")
    report.append(f"   Trip Distance: {df['trip_distance'].mean():.2f} miles")
    report.append(f"   Trip Duration: {df['trip_duration_minutes'].mean():.1f} minutes")
    report.append(f"   Fare Amount: ${df['fare_amount'].mean():.2f}")
    report.append(f"   Average Speed: {df['avg_speed_mph'].mean():.1f} mph")
    
    # Spatial insights
    report.append(f"\n🗺️ KEY SPATIAL INSIGHTS")
    report.append(f"   • Most trips originate in Manhattan zones")
    report.append(f"   • {df['is_manhattan_trip'].sum()/len(df)*100:.1f}% of trips are entirely within Manhattan")
    report.append(f"   • Rush hour speeds drop to {df[df['is_rush_hour']]['avg_speed_mph'].mean():.1f} mph")
    report.append(f"   • Airport trips represent ~{len(df[df['is_airport_pickup']])/len(df)*100:.1f}% of total volume")
    
    report.append("\n" + "="*70)
    return "\n".join(report)

print(generate_eda_report(df))

# Save to file option
# with open('nyc_taxi_eda_report.txt', 'w') as f:
#     f.write(generate_eda_report(df))

### Exercise 9 Solution: Predictive Features

In [ ]:
# Solution 9: Feature importance analysis
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import LabelEncoder

# Prepare features
features_df = df.copy()
le = LabelEncoder()
features_df['pickup_zone_encoded'] = le.fit_transform(features_df['pickup_zone'])
features_df['direction_encoded'] = le.fit_transform(features_df['direction_category'])

feature_cols = ['pickup_hour', 'pickup_latitude', 'pickup_longitude', 
               'dropoff_latitude', 'dropoff_longitude', 'pickup_zone_encoded',
               'is_weekend', 'is_rush_hour', 'pickup_dist_from_center',
               'direction_encoded', 'dist_from_jfk']

# Predict high fare (top 25%)
fare_threshold = df['fare_amount'].quantile(0.75)
features_df['high_fare'] = (features_df['fare_amount'] > fare_threshold).astype(int)

X = features_df[feature_cols]
y_fare = features_df['high_fare']
y_duration = features_df['trip_duration_minutes']
y_airport = features_df['is_airport_pickup'].astype(int)

# Train models
rf_fare = RandomForestRegressor(n_estimators=50, random_state=42)
rf_fare.fit(X, y_fare)

rf_duration = RandomForestRegressor(n_estimators=50, random_state=42)
rf_duration.fit(X, y_duration)

rf_airport = RandomForestRegressor(n_estimators=50, random_state=42)
rf_airport.fit(X, y_airport)

# Feature importance
importance_df = pd.DataFrame({
    'Feature': feature_cols,
    'High_Fare': rf_fare.feature_importances_,
    'Duration': rf_duration.feature_importances_,
    'Airport': rf_airport.feature_importances_
})

print("🤖 FEATURE IMPORTANCE ANALYSIS")
print("="*70)
print(importance_df.round(3))

# Visualize
importance_df.set_index('Feature').plot(kind='barh', figsize=(10, 8))
plt.title('Feature Importance for Different Predictions')
plt.xlabel('Importance')
plt.tight_layout()
plt.show()

print("\n💡 KEY FINDINGS:")
print("   • High Fare: Distance and location features most important")
print("   • Duration: Pickup hour (traffic) and distance crucial")
print("   • Airport: Distance from JFK and trip distance are key indicators")

### Exercise 10 Solution: Custom Zone Clustering

In [ ]:
# Solution 10: KMeans clustering for zones
from sklearn.cluster import KMeans

# Prepare coordinates for clustering
coords = df[['pickup_latitude', 'pickup_longitude']].copy()

# Fit KMeans
n_clusters = 8
kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
df['cluster_zone'] = kmeans.fit_predict(coords)

# Visualize clusters
plt.figure(figsize=(12, 8))
scatter = plt.scatter(df['pickup_longitude'], df['pickup_latitude'], 
                     c=df['cluster_zone'], cmap='tab10', alpha=0.5, s=10)
plt.colorbar(scatter, label='Cluster Zone')
plt.title('KMeans Clustering of Pickup Locations (8 Clusters)')
plt.xlabel('Longitude')
plt.ylabel('Latitude')
plt.xlim(-74.05, -73.75)
plt.ylim(40.60, 40.90)
plt.grid(True, alpha=0.3)
plt.show()

# Compare with original zones
print("🎯 CLUSTER VS ORIGINAL ZONES COMPARISON")
print("="*70)
comparison_table = pd.crosstab(df['cluster_zone'], df['pickup_zone'])
print(comparison_table)

# Cluster statistics
print("\n📊 CLUSTER STATISTICS")
print("="*70)
cluster_stats = df.groupby('cluster_zone')[['trip_distance', 'fare_amount', 'trip_duration_minutes']].mean().round(2)
cluster_stats['trip_count'] = df['cluster_zone'].value_counts().sort_index()
print(cluster_stats)

print("\n💡 INSIGHT: KMeans clusters follow natural density patterns;")
print("   may be more predictive than manual zones for ML models")

---

## 🎉 Congratulations! 

**Fantastic!** You have now gained valuable skills in geospatial data analysis — a powerful tool for real-world AI applications. Today you learned:

✅ How to visualize spatial distributions using scatter plots and heatmaps  
✅ Creating interactive maps with Plotly for exploratory analysis  
✅ Analyzing zone-based patterns and origin-destination flows  
✅ Combining temporal and spatial dimensions for rich insights  
✅ Engineering geospatial features for machine learning models  
✅ Detecting outliers and anomalies in spatial data  
✅ Understanding urban mobility patterns and rush hour dynamics  

### 🔮 Tomorrow's Preview
**Day 44: Comparative EDA — Titanic vs Spaceship Titanic**  
We'll compare two similar datasets from different contexts (historical vs futuristic) and learn how to perform comparative exploratory data analysis. You'll discover how to identify dataset-specific vs universal patterns!

---

*Python & AI Learning Path | Day 43 / 369*  
*Keep exploring, keep learning! 🚀*